# CALM — Adaptive Multimodal Wildfire Monitoring

**Hệ thống AI giám sát cháy rừng thích ứng đa phương thức**, theo kiến trúc agentic (URSA) và paper:

- *Adaptive Multimodal Wildfire Monitoring with Autonomous Agentic AI* (Agentic Remote Sensing)
- Kiến trúc: **Generator → Reflector → Formalizer** (3 node), **RSEN** (Reflexive Structured Experts Network), **Evidence Evaluator** (chống hallucination), **Safety Check** trước mỗi lệnh tool.

Notebook này minh họa: Planning Agent (Table A.1), RSEN (Equations 17–22), Wildfire QA, Full Pipeline (Plan + Evaluator)

## 0. Setup — Môi trường và LLM

In [1]:
# Đảm bảo có thể import calm (khi mở notebook từ thư mục calm/ hoặc myProject/)
import sys
from pathlib import Path
_root = Path.cwd()
if _root.name == "calm":
    _src = _root / "src"
else:
    _src = _root / "calm" / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Nạp biến môi trường từ .env
from calm.utils.env_loader import load_env
load_env(_root / ".env")
if _root.name != "calm" and (_root / "calm").exists():
    load_env(_root / "calm" / ".env")
else:
    load_env()


In [2]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    max_completion_tokens=10000,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
)

/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Planning Agent (URSA – 3-node, Table A.1)

Theo mô tả trong paper, **Planning Agent** của URSA gồm 3 thành phần chính:

1. **Generator**  
   - Nhận `user query`.  
   - Sinh ra kế hoạch thực hiện ở dạng ngôn ngữ tự nhiên.

2. **Reflector**  
   - Đánh giá kế hoạch do Generator tạo ra.  
   - Kiểm tra các tiêu chí:
     - **Clarity** (tính rõ ràng)
     - **Completeness** (tính đầy đủ)
     - **Feasibility** (tính khả thi)
     - **Safety** (tính an toàn)
   - Nếu kế hoạch đạt yêu cầu → trả về nhãn **`[APPROVED]`**.  
   - Nếu chưa đạt → phản hồi để Generator chỉnh sửa.

3. **Formalizer**  
   - Chuyển kế hoạch đã được duyệt thành định dạng **JSON `plan_steps`**.  
   - Quá trình có thể lặp lại tối đa **`f_max` lần thử** nếu việc chuẩn hóa thất bại.

> Prompt cho các thành phần này được thiết kế theo **Appendix A – Table A.1** của paper.

In [3]:
from calm.agents.planning_agent import PlanningAgent

agent = PlanningAgent(llm=llm, config={}, n_max=3, f_max=3)
query = "Wildfire risk assessment for Amazon region next 7 days"
result = agent.invoke(query)

plan = result.get("final_output") or []
print("Approved:", result.get("approved"))
print("Số bước kế hoạch:", len(plan))
for i, step in enumerate(plan, 1):
    print(f"  Bước {i}: {step.get('step_id', '?')} | action={step.get('action')} | agent={step.get('agent')}")

Approved: True
Số bước kế hoạch: 5
  Bước 1: step-1 | action=retrieve_weather_forecast | agent=qa
  Bước 2: step-2 | action=retrieve_historical_and_satellite_data | agent=data_knowledge
  Bước 3: step-3 | action=evaluate_firefighting_resources | agent=execution
  Bước 4: step-4 | action=invoke_prediction_model | agent=prediction
  Bước 5: step-5 | action=ensure_data_security | agent=qa


## 2. RSEN — Reflexive Structured Experts Network  
*(Equations 17–22, Appendix A Tables A.6–A.8)*

RSEN được thiết kế theo mô hình **đa chuyên gia phản tư (reflexive multi-expert system)**, trong đó các chuyên gia hoạt động **song song và độc lập** để phân tích các khía cạnh khác nhau của truy vấn.

### Kiến trúc thành phần

Hệ thống bao gồm ba tác nhân chính:

- **Weather Analyst**  
  Chuyên gia phân tích **khí tượng**, đánh giá các yếu tố liên quan đến thời tiết như:
  - nhiệt độ
  - lượng mưa
  - điều kiện khí hậu
  - các hiện tượng thời tiết đặc biệt

- **Geo Analyst**  
  Chuyên gia phân tích **địa lý**, xem xét các yếu tố:
  - đặc điểm địa hình
  - vị trí địa lý
  - điều kiện môi trường khu vực
  - tính hợp lý về mặt không gian

- **Ops Coordinator**  
  Tác nhân điều phối trung tâm có nhiệm vụ:
  - tổng hợp báo cáo từ các chuyên gia
  - đánh giá tính nhất quán giữa các phân tích
  - đưa ra quyết định cuối cùng:
    - **Plausible** — thông tin hợp lý  
    - **Implausible** — thông tin không hợp lý

### Cơ chế phản tư (Reflexion)

Hệ thống sử dụng **bộ nhớ dài hạn** dựa trên **ChromaDB**:

- **Collection:** `calm_rsen_memory`
- **Chức năng:** lưu trữ các trường hợp phân tích trước đó
- **Truy xuất:** sử dụng **top-k retrieval** để tìm các kinh nghiệm liên quan

Cơ chế này cho phép các chuyên gia:
- tham chiếu các phân tích trước đây
- cải thiện chất lượng lập luận
- tăng tính nhất quán và độ tin cậy của quyết định cuối cùng.

In [5]:
import warnings
warnings.filterwarnings("ignore", message="TqdmWarning|IProgress|BertModel|UNEXPECTED")

from calm.agents.rsen_module import RSENModule
from calm.memory.chroma_store import ChromaMemoryStore

memory = ChromaMemoryStore(
    collection_name="calm_rsen_memory",
    persist_directory=".chroma",
    k=3,
)
rsen = RSENModule(llm=llm, memory_store=memory, k=3)

result = rsen.validate(
    prediction={"risk_level": "High", "confidence": 0.8},
    met_data={"temperature": 35.0, "humidity": 0.2, "wind_speed": 15},
    spatial_data={"fuel_type": "Shrubland", "slope": 25, "elevation": 500},
)

print("Validation:", result.get("validation_decision"))
print("Rationale:", (result.get("final_rationale") or "")[:400])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9939.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/hungvu/code/khanh/v2/calm-/src/calm/memory/chroma_store.py:117: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='4c46b9ff-7e1d-412a-ba7c-bee95c9d40bd', metadata={}, page_content='Prediction: High. Validation: Plausible. Rationale: The weather report identifies extreme heat (35.0°C) and low humidity (20%), which significantly dry out fuels, increasing flammability. Moderate wind speeds can carry embers and enhance fire spread. Further, the geospatial report highlights that shrubland fuel type is easily ignitable, and the steep slope (25 degrees) promotes rapid upward fire movem

Validation: Plausible
Rationale: The meteorological conditions of high temperatures and low humidity are known to create an environment conducive to fire activity. This is compounded by the moderate wind speed, which can promote the spread of fires. The Geo-Spatial analysis identifies highly flammable shrubland and steep terrain, which together create a situation where fires can ignite and spread rapidly. Historical fire incident


## 3. Wildfire QA Agent — Evidence Evaluator  
*(Appendix A Tables A.2–A.5)*

Wildfire QA Agent là tác nhân hỏi–đáp chuyên biệt cho miền **cháy rừng**, được thiết kế theo nguyên tắc **evidence-driven reasoning** nhằm giảm thiểu **hallucination** của mô hình ngôn ngữ.

Hệ thống chỉ tạo câu trả lời khi có **bằng chứng hỗ trợ đầy đủ**, nếu không sẽ tiếp tục tìm kiếm dữ liệu bổ sung.

---

### 3.1 Kiến trúc Pipeline

Pipeline xử lý gồm các bước tuần tự sau:

1. **Retrieve — DataKnowledgeAgent**

   Truy vấn ban đầu được gửi tới **DataKnowledgeAgent** để thực hiện truy xuất tri thức từ các nguồn:

   - vector database (ChromaDB)
   - tài liệu tri thức nội bộ
   - bộ nhớ hệ thống (memory store)

   Kết quả là tập hợp **evidence candidates** liên quan đến câu hỏi.

---

2. **Evidence Evaluation**

   Tác nhân **Evidence Evaluator** đánh giá chất lượng và mức độ liên quan của các bằng chứng.

   Các tiêu chí đánh giá bao gồm:

   - độ liên quan ngữ nghĩa (semantic relevance)
   - độ tin cậy của nguồn
   - tính đầy đủ của thông tin
   - sự nhất quán giữa các bằng chứng

   Điểm đánh giá được so sánh với tham số:


In [6]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="duckduckgo_search")

from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.qa_agent import WildfireQAAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool

safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 5})
memory = ChromaMemoryStore(
    collection_name="calm_qa_memory",
    persist_directory=".chroma",
    k=3,
)
tools = {"earth_engine": None, "copernicus": None, "web_search": web, "arxiv": None}
data_agent = DataKnowledgeAgent(
    llm=llm, tools=tools, memory_store=memory, config={"dedup_check": True},
)
qa = WildfireQAAgent(
    llm=llm,
    data_agent=data_agent,
    web_search_tool=web,
    memory_store=memory,
    config={"evidence_threshold": 0.65},
)

result = qa.invoke("What caused the 2023 Canadian wildfires?")
out = result.get("final_output") or {}
print("Answer:", out.get("answer", out))
print("Citations:", out.get("citations", []))
print("Confidence:", out.get("confidence"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8026.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query, max_results=max_results))
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query, max_results=max_results))
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to

Answer: The 2023 Canadian wildfires were primarily driven by a combination of extreme climatic conditions and human factors. Key contributors include:

1. **Climatic Factors**: 
   - **Drought and Dry Conditions**: Many regions in Canada experienced unusually dry conditions leading up to the wildfire season. This was exacerbated by prolonged periods of below-average precipitation.
   - **Heatwaves**: The summer of 2023 saw record-breaking temperatures, which created an environment conducive to wildfire ignition and spread. Heatwaves significantly increased the flammability of vegetation.
   - **Lightning Strikes**: A significant proportion of wildfires ignited due to lightning strikes, which were frequent during thunderstorms in the region during the summer months.

2. **Human Activities**: 
   - **Arson and Accidental Ignitions**: There were reports of wildfires that were either set intentionally or ignited through human activities such as campfires or discarded cigarettes.
   - **Lan

## 4. Full Pipeline — Plan + LLM-as-a-Judge (Evaluator, §5.2)

Hệ thống sử dụng pipeline gồm hai thành phần chính: **Planning** và **Evaluation** nhằm đảm bảo chất lượng đầu ra của mô hình.

### 4.1 Planning

Từ **query** của người dùng, hệ thống tạo một **kế hoạch xử lý (plan)** mô tả các bước cần thực hiện, ví dụ:

- truy xuất dữ liệu (retrieval)
- phân tích thông tin
- gọi các công cụ cần thiết
- tổng hợp kết quả

Kế hoạch này giúp hệ thống xử lý truy vấn một cách có cấu trúc và logic.

### 4.2 Execution

Hệ thống thực thi kế hoạch đã tạo để sinh ra **candidate output** (kết quả hoặc câu trả lời ban đầu).

### 4.3 Evaluation — LLM-as-a-Judge

Kết quả đầu ra được chuyển đến **Evaluator Agent** để đánh giá chất lượng.  
Evaluator đóng vai trò **LLM-as-a-Judge**, chấm điểm câu trả lời theo **5 tiêu chí**, mỗi tiêu chí trong khoảng **0–100**:

| Tiêu chí | Mô tả |
|---|---|
| **Data Accuracy** | Độ chính xác và tính đúng đắn của dữ liệu |
| **Explainability** | Mức độ rõ ràng và khả năng giải thích của câu trả lời |
| **Jargon Avoidance** | Hạn chế sử dụng thuật ngữ chuyên môn khó hiểu |
| **Redundancy Avoidance** | Tránh lặp lại thông tin không cần thiết |
| **Citation Quality** | Chất lượng và độ tin cậy của các nguồn trích dẫn |

### 4.4 Passing Criterion

Điểm trung bình của năm tiêu chí được tính như sau:


average_score = mean(all_scores)


Một câu trả lời được chấp nhận khi:


average_score >= passing_score


Trong đó:


passing_score = 75

In [7]:
from calm.agents.evaluator_agent import EvaluatorAgent
from calm.agents.planning_agent import PlanningAgent

planner = PlanningAgent(llm=llm, config={})
evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})

query = "Assess wildfire risk for California Central Valley next 14 days"
plan_result = planner.invoke(query)
plan = plan_result.get("final_output") or []

eval_result = evaluator.evaluate(
    response={"plan": plan, "query": query},
    query=query,
)

print("Plan steps:", len(plan))
print("Evaluation passed:", eval_result.get("passed"))
print("Average score:", eval_result.get("average_score"))
print("Scores:", eval_result.get("scores", {}))
if eval_result.get("recommendations"):
    print("Recommendations:", eval_result["recommendations"])

Plan steps: 4
Evaluation passed: False
Average score: 63.0
Scores: {'data_accuracy': 70, 'explainability': 75, 'jargon_avoidance': 80, 'redundancy_avoidance': 90, 'citation_quality': 50}
Recommendations: ['Include specific data and sources to improve data accuracy and citation quality.', 'Provide a summary of findings or insights from the web search and knowledge retrieval steps to enhance explainability.', 'Consider using simpler language for any technical jargon to make it more accessible to non-experts.']
